In [2]:
import re

# 1. We simulate a perfectly extracted text from the physics PDF containing LaTeX tags
sample_physics_text = r"""
We can now apply Newton's second law of motion to this system. The fundamental principle states that the net force is equal to mass times acceleration.
Therefore, the equation of motion is:
$$ \sum \vec{F} = m \vec{a} $$
Where m is the mass of the object and a is the acceleration vector. 
If we consider the force of gravity, the gravitational force between two masses is defined as:
$$ F_g = G \frac{m_1 m_2}{r^2} $$
This equation is crucial for understanding planetary orbits. Now, let's move on to the concept of kinetic energy, which is given by the formula half mass times velocity squared.
"""

# ---------------------------------------------------------
# PIPELINE A: BASELINE CHUNKER (Standard character splitter)
# ---------------------------------------------------------
def baseline_chunker(text, chunk_size=120):
    """Splits text rigidly every N characters without caring about context."""
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

# ---------------------------------------------------------
# PIPELINE B: FORMULA-AWARE CHUNKER (Your Proposed Method)
# ---------------------------------------------------------
def formula_aware_chunker(text):
    """Splits text by paragraphs but ensures formulas ($$...$$) are never broken."""
    # Use regex to split the text keeping the LaTeX formulas intact
    # The parentheses in the regex keep the split pattern (the formula) in the list
    segments = re.split(r'(\$\$[\s\S]*?\$\$)', text)
    
    chunks = []
    current_chunk = ""
    
    for segment in segments:
        segment = segment.strip()
        if not segment:
            continue
            
        # If the segment is a formula, attach it to the current chunk safely
        if segment.startswith('$$') and segment.endswith('$$'):
            current_chunk += "\n" + segment + "\n"
        else:
            # If current chunk has content, save it before starting a new one
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = segment
            
    # Don't forget to add the last chunk
    if current_chunk:
        chunks.append(current_chunk.strip())
        
    return chunks

# ---------------------------------------------------------
# 3. RUN EXPERIMENT AND COMPARE
# ---------------------------------------------------------
print("=== PIPELINE A: BASELINE CHUNKING RESULT ===")
baseline_result = baseline_chunker(sample_physics_text)
for i, chunk in enumerate(baseline_result):
    print(f"--- Chunk {i+1} ---")
    print(chunk)

print("\n" + "="*50 + "\n")

print("=== PIPELINE B: FORMULA-AWARE CHUNKING RESULT ===")
formula_aware_result = formula_aware_chunker(sample_physics_text)
for i, chunk in enumerate(formula_aware_result):
    print(f"--- Chunk {i+1} ---")
    print(chunk)

=== PIPELINE A: BASELINE CHUNKING RESULT ===
--- Chunk 1 ---

We can now apply Newton's second law of motion to this system. The fundamental principle states that the net force is e
--- Chunk 2 ---
qual to mass times acceleration.
Therefore, the equation of motion is:
$$ \sum \vec{F} = m \vec{a} $$
Where m is the mas
--- Chunk 3 ---
s of the object and a is the acceleration vector. 
If we consider the force of gravity, the gravitational force between 
--- Chunk 4 ---
two masses is defined as:
$$ F_g = G \frac{m_1 m_2}{r^2} $$
This equation is crucial for understanding planetary orbits.
--- Chunk 5 ---
 Now, let's move on to the concept of kinetic energy, which is given by the formula half mass times velocity squared.



=== PIPELINE B: FORMULA-AWARE CHUNKING RESULT ===
--- Chunk 1 ---
We can now apply Newton's second law of motion to this system. The fundamental principle states that the net force is equal to mass times acceleration.
Therefore, the equation of motion is:
$$ \sum \vec{F